# Amber User Guide

**Version:** 24.1

**Audience:** Computational chemists and biophysicists using Amber for molecular dynamics (MD) simulations, binding affinities, and QM/MM calculations.

## TL;DR

This guide explains **how to use Amber effectively on this HPC system**.

**What is Amber?** Amber (Assisted Model Building with Energy Refinement) is a suite of programs for molecular dynamics simulations, particularly suited for biomolecular systems. It includes tools for system preparation (tleap), molecular dynamics (sander, pmemd), and binding free energy calculations (MMPBSA).

Run one simple case first:

```bash
# 1. Load the module
module load amber/24.1

# 2. Verify install
amber.python --version

# 3. Run a minimal command
tleap -h
```

It emphasizes:
- Quick setup for first-time users
- Choosing between CPU (sander) and GPU (pmemd) executables
- Common simulation workflows and troubleshooting

The goal is to **get you from zero to your first simulation in 20 minutes**, then move into production workflows.

## Table of Contents

- [TL;DR](#tldr)
- [PART I — Getting Started (Read Once)](#part-i--getting-started-read-once)
  - [TL;DR: 5-Minute QuickStart](#tldr-5-minute-quickstart)
  - [Step 1: Load the Module](#step-1-load-the-module)
  - [Step 2: Verify Setup](#step-2-verify-setup)
  - [Step 3: Run Your First Job](#step-3-run-your-first-job)
- [PART II — How to Use Amber (Read Carefully Once)](#part-ii--how-to-use-amber-read-carefully-once)
  - [What You Need to Know](#what-you-need-to-know)
  - [CPU vs GPU: When to Use Each](#cpu-vs-gpu-when-to-use-each)
  - [Resource Requests](#resource-requests)
- [PART III — Workflows (Use As Needed)](#part-iii--workflows-use-as-needed)
  - [Workflow 1: System Preparation with tleap](#workflow-1-system-preparation-with-tleap)
  - [Workflow 2: CPU MD Simulation (SLURM Batch - Build Partition)](#workflow-2-cpu-md-simulation-slurm-batch---build-partition)
  - [Workflow 3: GPU MD Simulation (SLURM Batch - GPU Partition)](#workflow-3-gpu-md-simulation-slurm-batch---gpu-partition)
- [PART IV — Troubleshooting (Skim When Broken)](#part-iv--troubleshooting-skim-when-broken)
  - [Symptom-Based Lookup](#symptom-based-lookup)
  - [FAQ](#faq)
  - [External Resources](#external-resources)

_💡 Tip: Enable nbextensions like "Collapsible Headings" and "Table of Contents (2)" for better navigation._

---

## PART I — Getting Started (Read Once)

**Purpose:** "I just want this to work."

This part should feel **quick and confidence-building**. By the end, you'll have a working setup and have run your first command successfully. You don't need to understand everything yet—just follow the steps.

### TL;DR: 5-Minute QuickStart

If you just want to run a command:

```bash
# 1. Load the module
module load amber/24.1

# 2. Verify it worked
amber.python --version

# 3. Check core tools
tleap -h
sander -h 2>&1 | head -20
```

**Expected result:** You should see version information and help text for Amber's core tools.

---

### Step 1: Load the Module

All software on this system is pre-installed and managed through the **module system** (lmod). To get started, load Amber:

In [ ]:
# List available versions (optional)
module avail amber

**Expected output:**
```
amber/22.01     amber/23.04     amber/24.1 (default)
```

Now load the module:

In [ ]:
# Load the default version
module load amber

# Or load a specific version
module load amber/24.1

**Expected output:** No output on success. (Use Step 2 below to verify.)

---

### Step 2: Verify Setup

Confirm the module loaded correctly and your environment is ready:

In [ ]:
# Check that the module is now loaded
module list

**Expected output:**
```
Currently Loaded Modules:
  1) amber/24.1
```

Check the Amber Python environment:

In [ ]:
# Verify Amber tools are accessible
amber.python --version

**Expected output:**
```
Python 3.11.x :: Amber distribution
```

✅ **Success!** Your environment is ready to use.

---

### Step 3: Run Your First Job

Here's the simplest possible workflow to confirm everything works end-to-end:

In [ ]:
# Check which tools are available
module load amber
echo "=== tleap (topology preparation) ==="
tleap -h 2>&1 | head -15

**Expected output:**
```
=== tleap (topology preparation) ===
tleap is a fully functional shell-like interface to Amber's leap
...
```

Now check the main simulation engines:

In [ ]:
# Check CPU simulation engine (sander)
echo "=== sander (CPU MD engine) ==="
sander -h 2>&1 | head -5

# Check GPU simulation engine (pmemd.cuda)
echo ""
echo "=== pmemd.cuda (GPU MD engine) ==="
if command -v pmemd.cuda &> /dev/null; then
    echo "pmemd.cuda is available"
else
    echo "pmemd.cuda not in PATH (check #SBATCH --gres=gpu)"
fi

**Expected output:**
```
=== sander (CPU MD engine) ===
| Sanding Amber off with sander!

=== pmemd.cuda (GPU MD engine) ===
pmemd.cuda is available
```

🎉 **You're set up!** Continue to Part II to understand how to use the system, or jump to Part III to see common workflows for your task.

**Next Steps:**
- **Want to understand how resources work?** Continue to [Part II](#part-ii--how-to-use-amber-read-carefully-once) below.
- **Ready for real workflows?** Jump to [Part III](#part-iii--workflows-use-as-needed) to find your task.
- **Hit an error?** Go to [Part IV Troubleshooting](#part-iv--troubleshooting-skim-when-broken).

---

## PART II — How to Use Amber (Read Carefully Once)

**Purpose:** "Help me not screw this up."

This part explains key concepts and decision-making. You'll understand **when** to use different options and **why** certain choices matter.

### What You Need to Know

#### Amber Tools Overview

| Tool | Purpose | When to Use |
|------|---------|-------------|
| **tleap** | System preparation (topology, coordinates) | Always first—before any MD run |
| **sander** | CPU-based MD engine | Testing, small systems, debugging |
| **pmemd** | GPU-accelerated MD engine | Production runs with large systems |
| **MMPBSA.py** | Binding affinity & energy decomposition | MM-PBSA and MM-GBSA calculations |
| **antechamber** | Parameter generation for small molecules | Before running QM/MM or non-standard residues |

#### Input and Output Files

**Input files (required for sander/pmemd):**
- `.prmtop` — Amber topology file (created by tleap)
- `.inpcrd` or `.rst7` — Initial coordinates file (created by tleap)
- `.mdin` — Simulation control file (you create this)

**Output files (standard):**
- `.mdout` — Main output log
- `.mdcrd` — Trajectory (coordinates every N steps)
- `.rst7` — Final restart file (for next simulation stage)
- `.nfo` — Information file

#### Supported Platforms

| Platform | Status | Notes |
|----------|--------|-------|
| CPU (sander) | ✅ Full support | Good for testing, all features available |
| GPU (pmemd.cuda) | ✅ Full support | ~50x faster for large systems |
| GPU (pmemd.hip) | ⚠️ Limited | AMD GPU support; check availability |

### CPU vs GPU: When to Use Each

**Use CPU (sander) if:**
- Your system is small (<5,000 atoms)
- You're testing a new simulation setup
- You need debugging output
- GPU nodes are full (faster queue time)

**Use GPU (pmemd.cuda) if:**
- Your system is large (>5,000 atoms)
- You're running production simulations
- You need results quickly (50x speedup typical)
- You can wait for GPU availability

#### Decision Guide

```
Is your system smaller than 5,000 atoms?
│
├── Yes → Use CPU sander (faster queue, good enough)
│
└── No
    │
    ├── Do you have time to wait?
    │     └── Yes → Use CPU sander (saves GPU for others)
    │
    └── No → Use GPU pmemd.cuda (50x faster)
```

### Resource Requests

**Don't request more than you need.**

- **CPU jobs (sander):** 4-16 cores, 8-32 GB RAM (depends on system size)
- **GPU jobs (pmemd.cuda):** 1-2 GPUs, 16-64 GB RAM, 1-4 CPU cores

**Rule of thumb:**
- ~1 MB per atom in RAM (rough estimate)
- Trajectory output: ~1-2 MB per 1000 steps for large systems

For questions about resources, see Part IV FAQ.

---

## PART III — Workflows (Use As Needed)

**Purpose:** "Show me how to actually do science."

Pick the workflow that matches your task. Each is self-contained.

### Workflow 1: System Preparation with tleap

Before running any MD simulation, you must prepare your system using **tleap**. This step generates the `.prmtop` (topology) and `.rst7` (coordinates) files.

**Step 1: Create a tleap input script**

In [ ]:
# Create a sample tleap script for a small peptide
cat > leaprc.input << 'EOF'
# Load standard force field
source leaprc.ff14SB

# Load water model
source leaprc.water.tip3p

# Load your PDB file
mol = loadpdb my_protein.pdb

# Check for missing atoms
check mol

# Solvate the system (10 Angstrom buffer)
solvated = solvateBox mol TIP3PBOX 10.0

# Add ions to neutralize
addions solvated Na+ 0

# Save topology and coordinate files
saveamberparm solvated system.prmtop system.inpcrd

# Quit tleap
quit
EOF
cat leaprc.input

**Expected output:**
```
# Load standard force field
source leaprc.ff14SB
...
quit
```

**Step 2: Run tleap to prepare the system**

In [ ]:
module load amber

# Run tleap (this is a dry-run with a dummy PDB)
echo "Note: This example requires a valid my_protein.pdb file."
echo "Running tleap would look like this:"
echo ""
echo "tleap -f leaprc.input"

**Expected output during a real run:**
```
Welcome to LEaP, version X.XX
...
Checking the system ...
...
Created topology file system.prmtop
Created coordinate file system.inpcrd
```

✅ **Success!** You now have `system.prmtop` and `system.inpcrd` ready for MD simulations.

---

### Workflow 2: CPU MD Simulation (SLURM Batch - Build Partition)

This workflow runs a molecular dynamics simulation on CPU cores using the build partition. Suitable for testing and small to medium systems.

**Step 1: Create a simulation control file (.mdin)**

In [ ]:
# Create a simple MD input file for equilibration
cat > minimize_and_heat.mdin << 'EOF'
Minimization and heating of structure
 &cntrl
  ! Control flags
  imin=1,                  ! Perform minimization
  ntx=1, irest=0,          ! New MD simulation
  
  ! Minimization parameters
  maxcyc=1000,             ! Max minimization steps
  ncyc=100,                ! Steepest descent steps
  
  ! System definition
  ntpr=50,                 ! Print frequency
  ntwr=1000,               ! Restart file frequency
  ntwe=0,                  ! Dont write energy file
  ntwx=100,                ! Trajectory frequency
  
  ! Non-bonded interactions
  cut=8.0,                 ! Cut-off distance (Angstroms)
  
  ! Force field
  ntb=1,                   ! Periodic boundary (constant volume)
  ntp=0,                   ! No pressure scaling
  ntwv=0, ioutfm=1,
  idip=0, ityp=0,
 /
EOF
cat minimize_and_heat.mdin

**Expected output:**
```
Minimization and heating of structure
 &cntrl
  imin=1,
  ...
 /
```

**Step 2: Create SLURM batch script for CPU partition**

In [ ]:
# Create CPU SLURM batch script
cat > amber_cpu.slurm << 'EOF'
#!/bin/bash
#SBATCH --job-name=amber_md_cpu
#SBATCH --partition=build              # Use build partition
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=16           # CPU cores
#SBATCH --cpus-per-task=1
#SBATCH --mem=32G                      # Memory allocation
#SBATCH --time=24:00:00                # Wall-clock time (24 hours)
#SBATCH --output=amber_cpu.log
#SBATCH --error=amber_cpu.err

# Load Amber module
module load amber/24.1

# Set simulation parameters
PRMTOP="system.prmtop"
INPCRD="system.inpcrd"
MDIN="minimize_and_heat.mdin"
MDOUT="minimize_and_heat.mdout"
MDCRD="minimize_and_heat.mdcrd"
RESTRT="minimize_and_heat.rst7"

# Verify input files exist
if [[ ! -f $PRMTOP ]] || [[ ! -f $INPCRD ]] || [[ ! -f $MDIN ]]; then
    echo "ERROR: Missing input files (prmtop, inpcrd, or mdin)"
    exit 1
fi

echo "Starting Amber MD (CPU sander) on $(date)"
echo "Job ID: $SLURM_JOB_ID"
echo "Partition: build"
echo "Cores: $SLURM_NTASKS"
echo "Memory: $SLURM_MEM_PER_NODE"
echo ""

# Run CPU simulation with mpirun for better performance
sander -O -i $MDIN -o $MDOUT -p $PRMTOP -c $INPCRD -r $RESTRT -x $MDCRD

if [[ $? -eq 0 ]]; then
    echo ""
    echo "Amber simulation completed successfully on $(date)"
    echo "Output files:"
    echo "  - $MDOUT (main output log)"
    echo "  - $MDCRD (trajectory)"
    echo "  - $RESTRT (final restart file)"
else
    echo "ERROR: Amber simulation failed!"
    exit 1
fi
EOF
cat amber_cpu.slurm

**Expected output:**
```
#!/bin/bash
#SBATCH --job-name=amber_md_cpu
#SBATCH --partition=build
...
```

**Step 3: Submit and monitor the job**

In [ ]:
# Make the script executable
chmod +x amber_cpu.slurm

# Submit to SLURM (example only—requires compute access)
echo "To submit the job, run:"
echo "  sbatch amber_cpu.slurm"
echo ""
echo "To monitor progress:"
echo "  squeue -u \$USER"
echo "  tail -f amber_cpu.log"
echo ""
echo "To check job details:"
echo "  scontrol show job <JOB_ID>"

**Expected workflow:**
```bash
$ sbatch amber_cpu.slurm
Submitted batch job 12345

$ squeue -u $USER
JOBID PARTITION NAME            USER ST TIME NODES CPUS
12345 build     amber_md_cpu   user R  5:32 1     16

$ tail -f amber_cpu.log
Starting Amber MD (CPU sander) on Tue Apr 2 10:15:42 UTC 2026
Job ID: 12345
Partition: build
Cores: 16
Memory: 32G
```

🎉 **Job submitted!** The simulation will run on 16 CPU cores in the build partition. Check computational efficiency by monitoring timing in `minimize_and_heat.mdout`.

**Performance notes for CPU (sander):**
- Typical speed: 10-30 ns/day (depends on system size and CPU architecture)
- Scaling: Good up to ~32 cores; diminishing returns beyond that
- Best for: Testing, debugging, small systems (<10,000 atoms)

---

### Workflow 3: GPU MD Simulation (SLURM Batch - GPU Partition)

This workflow runs a molecular dynamics simulation on GPU accelerators. Expect 50x speedup over CPU for large systems.

**Step 1: Create a production MD control file**

In [ ]:
# Create a production MD input file
cat > production_md.mdin << 'EOF'
Production MD simulation with NPT ensemble
 &cntrl
  ! Simulation type
  imin=0,                  ! MD (not minimization)
  ntx=5, irest=1,          ! Read velocities from restart
  ig=-1,                   ! Random seed
  
  ! Time and output
  nstlim=50000,            ! Number of MD steps
  dt=0.002,                ! Time step (2 fs)
  ntpr=500,                ! Print frequency
  ntwr=5000,               ! Restart frequency
  ntwx=100,                ! Trajectory frequency
  ntwe=0,                  ! No energy file
  
  ! Thermostat (Berendsen)
  ntt=3,                   ! Langevin thermostat
  temp0=298.15, tempi=298.15,
  gamma_ln=1.0,            ! Collision frequency
  
  ! Barostat (Berendsen)
  ntp=1,                   ! Isotropic pressure scaling
  pres0=1.0,
  taup=2.0,
  
  ! Non-bonded interactions
  cut=8.0,
  ntb=2,                   ! Periodic boundary (NPT)
  
  ! Force field options
  icfe=0, ifmbar=0,
  ioutfm=1, iwrap=1,
 /
EOF
cat production_md.mdin

**Expected output:**
```
Production MD simulation with NPT ensemble
 &cntrl
  imin=0,
  ...
 /
```

**Step 2: Create SLURM batch script for GPU partition**

In [ ]:
# Create GPU SLURM batch script
cat > amber_gpu.slurm << 'EOF'
#!/bin/bash
#SBATCH --job-name=amber_md_gpu
#SBATCH --partition=gpu                # Use GPU partition
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=2            # 2 CPU tasks for GPU node
#SBATCH --cpus-per-task=4              # 4 CPUs per task
#SBATCH --gres=gpu:2                   # Request 2 GPUs
#SBATCH --mem=64G                      # Memory allocation
#SBATCH --time=48:00:00                # Wall-clock time (48 hours)
#SBATCH --output=amber_gpu.log
#SBATCH --error=amber_gpu.err

# Load Amber module
module load amber/24.1

# Verify GPU availability
if ! command -v pmemd.cuda &> /dev/null; then
    echo "ERROR: pmemd.cuda not found. Check Amber installation or GPU support."
    exit 1
fi

# Set simulation parameters
PRMTOP="system.prmtop"
INPCRD="system.inpcrd"
MDIN="production_md.mdin"
MDOUT="production_md.mdout"
MDCRD="production_md.mdcrd"
RESTRT="production_md.rst7"

# Verify input files
if [[ ! -f $PRMTOP ]] || [[ ! -f $INPCRD ]] || [[ ! -f $MDIN ]]; then
    echo "ERROR: Missing input files (prmtop, inpcrd, or mdin)"
    exit 1
fi

# Set GPU devices
export CUDA_VISIBLE_DEVICES=0,1

echo "Starting Amber MD (GPU pmemd.cuda) on $(date)"
echo "Job ID: $SLURM_JOB_ID"
echo "Partition: gpu"
echo "GPUs: 2 (NVIDIA)"
echo "CPU cores: $((SLURM_NTASKS * SLURM_CPUS_PER_TASK))"
echo "Memory: $SLURM_MEM_PER_NODE"
echo ""

# Check GPU status (for diagnostics)
echo "GPU Status:"
nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv,noheader
echo ""

# Run GPU simulation
# pmemd.cuda launches on all available CUDA devices (controlled by CUDA_VISIBLE_DEVICES)
time pmemd.cuda -O -i $MDIN -o $MDOUT -p $PRMTOP -c $INPCRD -r $RESTRT -x $MDCRD

if [[ $? -eq 0 ]]; then
    echo ""
    echo "Amber GPU simulation completed successfully on $(date)"
    echo "Output files:"
    echo "  - $MDOUT (main output log)"
    echo "  - $MDCRD (trajectory)"
    echo "  - $RESTRT (final restart file)"
    
    # Extract timing from output
    echo ""
    echo "Performance summary:"
    grep -A 5 "Performance Summary" $MDOUT || grep "ns/day" $MDOUT
else
    echo "ERROR: Amber GPU simulation failed!"
    echo "Check CUDA device availability and driver installation."
    exit 1
fi
EOF
cat amber_gpu.slurm

**Expected output:**
```
#!/bin/bash
#SBATCH --job-name=amber_md_gpu
#SBATCH --partition=gpu
...
```

**Step 3: Submit and monitor the GPU job**

In [ ]:
# Make the script executable
chmod +x amber_gpu.slurm

# Submit to SLURM (example only—requires compute access)
echo "To submit the job, run:"
echo "  sbatch amber_gpu.slurm"
echo ""
echo "To monitor GPU usage in real-time:"
echo "  squeue -u \$USER"
echo "  ssh node-name nvidia-smi -l 1  # Monitor GPU on compute node"
echo ""
echo "To check job status:"
echo "  scontrol show job <JOB_ID>"
echo ""
echo "To watch the output log:"
echo "  tail -f amber_gpu.log"

**Expected workflow:**
```bash
$ sbatch amber_gpu.slurm
Submitted batch job 54321

$ squeue -u $USER
JOBID PARTITION NAME            USER ST TIME NODES CPUS GRES(IDX)
54321 gpu       amber_md_gpu   user R  10:15 1    2  gpu:2(0,1)

$ tail -f amber_gpu.log
Starting Amber MD (GPU pmemd.cuda) on Tue Apr 2 14:30:22 UTC 2026
Job ID: 54321
Partition: gpu
GPUs: 2 (NVIDIA)
CPU cores: 8
Memory: 64G
...
GPU Status:
0, NVIDIA A100, 40960 MB, 525.105.17
1, NVIDIA A100, 40960 MB, 525.105.17
```

⚡ **GPU job running!** Expect ~5-15 μs/day (wall-clock) compared to ~0.1-0.3 μs/day on CPU (50x faster).

**Performance notes for GPU (pmemd.cuda):**
- Typical speed: 5-15 ns/day (per GPU pair, system dependent)
- Scaling: Excellent up to 4 GPUs; diminishing returns beyond that
- Best for: Production runs, large systems (>20,000 atoms)
- GPU memory: NVIDIAs A100/H100 supported; typical requirement 2-8 GB per GPU

---

## PART IV — Troubleshooting (Skim When Broken)

**Purpose:** "Something failed. Give me answers fast."

### Symptom-Based Lookup

| Symptom | Likely Cause | Fix |
|---------|--------------|-----|
| `command: amber.python not found` | Module not loaded | `module load amber/24.1` |
| `tleap: command not found` | Path issue with module | Check `which tleap` and reload module |
| `CUDA out of memory in pmemd.cuda` | System too large for GPU | Use multiple GPUs or switch to CPU (sander) |
| `pmemd.cuda: CUDA error: invalid device ordinal` | GPU not accessible | Check `nvidia-smi` on compute node; verify `--gres=gpu` in SLURM script |
| `ERROR: Could not find a matching topology type` | Invalid force field in tleap | Verify `source leaprc.ffXX` matches your residues |
| `sander: Segmentation fault` | Corrupted topology file | Regenerate `.prmtop` with tleap |
| `WARNING: PSF-style elements not recognized` | Atom type issue | Check PDB file; may need antechamber for non-standard residues |
| Simulation exits early without error | Stability issue (bad coordinates) | Check initial structure; try minimization with higher `maxcyc` |

### FAQ

**Q: How do I check my job status?**

A: Use `squeue -u $USER` to see running/queued jobs. For detailed info: `scontrol show job <JOB_ID>`.

**Q: Can I run multiple simulations at once?**

A: Yes! Submit multiple SLURM jobs. Check your cluster's job limit policy with `sinfo` or contact support.

**Q: How do I extract coordinates from the trajectory?**

A: Use the AMBER tool `cpptraj`:
```bash
cpptraj -p system.prmtop -i commands.in
# Where commands.in contains:
# trajin production_md.mdcrd
# strip :WAT
# autoimage
# trajout cleaned.mdcrd
# go
```

**Q: My system has non-standard residues. What do I do?**

A: Use `antechamber` to generate GAFF parameters:
```bash
antechamber -i ligand.mol2 -fi mol2 -o ligand.gaff.mol2 -fo mol2 -c bcc
parmchk2 -i ligand.gaff.mol2 -f mol2 -o ligand.gaff.frcmod
```

**Q: How do I calculate binding free energy (MM-PBSA)?**

A: After production MD, use:
```bash
MMPBSA.py -O -i mmpbsa.in -o results.dat \\
    -p system.prmtop -y production_md.mdcrd -c system.inpcrd
```

**Q: My GPU job won't start. What gives?**

A: Common issues:
- GPU not in `--gres=gpu:N` requested: Add to SLURM script
- CUDA not available: Check with `nvidia-smi` on login node
- Driver mismatch: Contact HPC support; pmemd.cuda requires specific NVIDIA driver versions

### External Resources

- **Amber Official Documentation:** [https://ambermd.org/](https://ambermd.org/)
- **Amber Tutorials:** [https://ambermd.org/tutorials/](https://ambermd.org/tutorials/)
- **AMBER ForceFields:** [https://ambermd.org/forcefields/](https://ambermd.org/forcefields/)
- **GPU Performance Guide:** [https://ambermd.org/gpu/](https://ambermd.org/gpu/)
- **GROMACS Interoperability:** Convert Amber files with `acpype` or `ParmEd`
- **Community Forum:** [https://forums.ambermd.org/](https://forums.ambermd.org/)
- **Cite Amber:** Case DA, et al. *J. Chem. Theory Comput.* 2020, 16, 1 (and respective modules)